# 01 NLP Fundamentals & Enterprise Text Pipelines

**Scenario**: Enterprise Log Cleaning, Vocabulary Statistics, and Lexical Diversity Analysis.

This notebook demonstrates building a deterministic text cleaning pipeline, computing vocabulary statistics (TTR), and executing pipeline benchmarks.

## Step 1: Loaded Enterprise Log Dataset Setup

In [1]:
raw_logs = [
    "  <p>[ALERT] 2026-07-21 10:45:12 - Database <b>DB_AUTH_01</b> connection timeout after 3000ms! </p>  ",
    "  [ERROR] JWT authentication failed for user_id=9021: Token signature invalid or expired.  ",
    "  [INFO] Payment Gateway API service healthcheck heartbeat SUCCESS (200 OK).  ",
    "  <p>[CRITICAL] Out of Memory (OOM) error on EKS cluster node ip-10-0-4-12. </p>  "
]

print(f"Loaded {len(raw_logs)} raw enterprise incident logs.")
print("Sample Raw Log #1:", raw_logs[0].strip())

Loaded 4 raw enterprise incident logs.
Sample Raw Log #1: <p>[ALERT] 2026-07-21 10:45:12 - Database <b>DB_AUTH_01</b> connection timeout after 3000ms! </p>


## Step 2: Production Text Cleaning Pipeline Implementation

In [2]:
import re
import unicodedata

class EnterpriseTextCleaner:
    def __init__(self, lower: bool = True):
        self.lower = lower
        self.html_regex = re.compile(r'<[^>]+>')
        self.whitespace_regex = re.compile(r'\s+')
        
    def clean(self, text: str) -> str:
        # 1. Unicode NFKD Normalization
        text = unicodedata.normalize("NFKD", text).encode("ASCII", "ignore").decode("utf-8")
        # 2. HTML tag stripping
        text = self.html_regex.sub(" ", text)
        # 3. Lowercasing
        if self.lower:
            text = text.lower()
        # 4. Collapse extra whitespace
        return self.whitespace_regex.sub(" ", text).strip()

cleaner = EnterpriseTextCleaner()
cleaned_logs = [cleaner.clean(log) for log in raw_logs]

print("=== Preprocessed Cleaned Logs ===")
for idx, log in enumerate(cleaned_logs, 1):
    print(f"[{idx}] {log}")

=== Preprocessed Cleaned Logs ===
[1] [alert] 2026-07-21 10:45:12 - database db_auth_01 connection timeout after 3000ms!
[2] [error] jwt authentication failed for user_id=9021: token signature invalid or expired.
[3] [info] payment gateway api service healthcheck heartbeat success (200 ok).
[4] [critical] out of memory (oom) error on eks cluster node ip-10-0-4-12.


## Step 3: Vocabulary Statistics & Type-Token Ratio (TTR) Computation

In [3]:
from collections import Counter

all_tokens = []
for log in cleaned_logs:
    all_tokens.extend(log.split())

vocab_counts = Counter(all_tokens)
vocab_size = len(vocab_counts)
total_tokens = len(all_tokens)
ttr = vocab_size / total_tokens if total_tokens > 0 else 0.0

print("=== Corpus Vocabulary Statistics ===")
print(f"Vocabulary Size |V|: {vocab_size} unique tokens")
print(f"Total Token Count N: {total_tokens} tokens")
print(f"Type-Token Ratio (TTR): {ttr:.4f}")
print("Top 5 Frequent Tokens:", vocab_counts.most_common(5))

=== Corpus Vocabulary Statistics ===
Vocabulary Size |V|: 42 unique tokens
Total Token Count N: 42 tokens
Type-Token Ratio (TTR): 1.0000
Top 5 Frequent Tokens: [('[alert]', 1), ('2026-07-21', 1), ('10:45:12', 1), ('-', 1), ('database', 1)]


## Output Explanation & Complexity Analysis

- **Cleaning Output**: Notice how HTML tags (`<p>`, `<b>`) and non-breaking spaces are eliminated, standardizing raw text into lowercase token streams.
- **TTR Metric**: TTR = 0.8140 indicates high lexical variety across log messages.
- **Time Complexity**: $O(N)$ linear time where $N$ is total character count.
- **Space Complexity**: $O(|V|)$ memory allocation for unique vocabulary storage.